# 6A组 白盒对抗样本攻防实验 - 交互式演示

本笔记本提供对抗攻击与防御的直观可视化演示，无需完整训练即可查看核心效果。
**运行前请确保已执行过 `python train.py` 生成了预训练模型文件**

In [ ]:
# 导入依赖
import torch
import numpy as np
import matplotlib.pyplot as plt
from data_utils import *
from model import ResNet18_CIFAR
from eval import fgsm_attack, pgd_attack, deepfool_attack, JPEGDefense, evaluate_clean, test_attack

# 全局设置
plt.rcParams['font.sans-serif'] = ['SimHei']  # 解决中文显示
plt.rcParams['axes.unicode_minus'] = False

# 加载数据和模型
trainloader, testloader = get_dataloaders()

# 加载干净模型（路径适配code外的models文件夹）
model_clean = ResNet18_CIFAR().to(device)
model_clean.load_state_dict(torch.load("../models/resnet18_clean.pth", map_location=device))
model_clean.eval()

# 加载对抗训练模型
model_adv = ResNet18_CIFAR().to(device)
model_adv.load_state_dict(torch.load("../models/resnet18_adv.pth", map_location=device))
model_adv.eval()

print(f"✅ 模型加载成功，运行设备: {device}")
print(f"✅ 干净模型准确率: {evaluate_clean(model_clean, testloader, device):.2f}%")
print(f"✅ 对抗训练模型准确率: {evaluate_clean(model_adv, testloader, device):.2f}%")

---
## 1. 三种白盒攻击效果可视化
对比原始样本、FGSM、PGD、DeepFool攻击后的样本及预测结果

In [ ]:
# 随机抽取5个测试样本
num_samples = 5
images, labels = next(iter(testloader))
images, labels = images[:num_samples].to(device), labels[:num_samples].to(device)

# 生成三种对抗样本（参数和全局配置一致）
fgsm_imgs = fgsm_attack(model_clean, images, labels, epsilon=epsilon)
pgd_imgs = pgd_attack(model_clean, images, labels, epsilon=epsilon, alpha=pgd_alpha, steps=pgd_steps)
deepfool_imgs = deepfool_attack(model_clean, images, max_iter=deepfool_max_iter, overshoot=deepfool_overshoot)

# 获取预测结果
with torch.no_grad():
    orig_preds = model_clean(images).argmax(1)
    fgsm_preds = model_clean(fgsm_imgs).argmax(1)
    pgd_preds = model_clean(pgd_imgs).argmax(1)
    deep_preds = model_clean(deepfool_imgs).argmax(1)

# 绘制对比图
fig, axes = plt.subplots(num_samples, 4, figsize=(12, 3*num_samples))
if num_samples == 1:
    axes = axes[None, :]

titles = ['原始样本', 'FGSM攻击', 'PGD攻击', 'DeepFool攻击']
all_imgs = [images, fgsm_imgs, pgd_imgs, deepfool_imgs]
all_preds = [orig_preds, fgsm_preds, pgd_preds, deep_preds]

for i in range(num_samples):
    true_label = classes[labels[i]]
    for j in range(4):
        img = all_imgs[j][i].cpu().permute(1,2,0).numpy()
        pred_label = classes[all_preds[j][i]]
        color = 'green' if pred_label == true_label else 'red'
        
        axes[i,j].imshow(np.clip(img, 0, 1))
        axes[i,j].set_title(f'{titles[j]}\n预测: {pred_label}\n真实: {true_label}', color=color)
        axes[i,j].axis('off')

plt.tight_layout()
plt.savefig('../plots/demo_attacks.png', dpi=200)
plt.show()

---
## 2. 防御方法效果演示
### 2.1 JPEG压缩防御效果

In [ ]:
# 对PGD对抗样本进行JPEG压缩净化
jpeg_def = JPEGDefense(quality=jpeg_quality)
purified_imgs = jpeg_def(pgd_imgs)

# 获取净化后的预测结果
with torch.no_grad():
    purified_preds = model_clean(purified_imgs).argmax(1)

# 绘制对比图
fig, axes = plt.subplots(num_samples, 3, figsize=(9, 3*num_samples))
if num_samples == 1:
    axes = axes[None, :]

titles = ['原始样本', 'PGD攻击样本', 'JPEG净化后']
all_imgs = [images, pgd_imgs, purified_imgs]
all_preds = [orig_preds, pgd_preds, purified_preds]

for i in range(num_samples):
    true_label = classes[labels[i]]
    for j in range(3):
        img = all_imgs[j][i].cpu().permute(1,2,0).numpy()
        pred_label = classes[all_preds[j][i]]
        color = 'green' if pred_label == true_label else 'red'
        
        axes[i,j].imshow(np.clip(img, 0, 1))
        axes[i,j].set_title(f'{titles[j]}\n预测: {pred_label}\n真实: {true_label}', color=color)
        axes[i,j].axis('off')

plt.tight_layout()
plt.savefig('../plots/demo_jpeg_defense.png', dpi=200)
plt.show()

### 2.2 对抗训练模型鲁棒性对比

In [ ]:
# 用对抗训练模型预测相同的PGD对抗样本
with torch.no_grad():
    adv_model_preds = model_adv(pgd_imgs).argmax(1)

# 绘制对比图
fig, axes = plt.subplots(num_samples, 3, figsize=(9, 3*num_samples))
if num_samples == 1:
    axes = axes[None, :]

titles = ['原始样本', 'PGD攻击样本', '对抗训练模型预测']
all_imgs = [images, pgd_imgs, pgd_imgs]
all_preds = [orig_preds, pgd_preds, adv_model_preds]

for i in range(num_samples):
    true_label = classes[labels[i]]
    for j in range(3):
        img = all_imgs[j][i].cpu().permute(1,2,0).numpy()
        pred_label = classes[all_preds[j][i]]
        color = 'green' if pred_label == true_label else 'red'
        
        axes[i,j].imshow(np.clip(img, 0, 1))
        axes[i,j].set_title(f'{titles[j]}\n预测: {pred_label}\n真实: {true_label}', color=color)
        axes[i,j].axis('off')

plt.tight_layout()
plt.savefig('../plots/demo_adv_training.png', dpi=200)
plt.show()

---
## 3. 攻击参数交互测试
调节FGSM的扰动大小ε，观察攻击效果的变化

In [ ]:
# 测试不同epsilon值的FGSM攻击效果
test_epsilons = [0.005, 0.01, 0.02, 0.04, 0.08, 0.16]
test_img_idx = 0  # 选择第0个样本进行测试

test_img = images[test_img_idx:test_img_idx+1].to(device)
test_label = labels[test_img_idx:test_img_idx+1].to(device)
true_label = classes[test_label.item()]

# 生成不同epsilon的对抗样本
adv_imgs_list = []
preds_list = []

for eps in test_epsilons:
    adv_img = fgsm_attack(model_clean, test_img, test_label, epsilon=eps)
    with torch.no_grad():
        pred = model_clean(adv_img).argmax(1).item()
    adv_imgs_list.append(adv_img)
    preds_list.append(pred)

# 绘制对比图
fig, axes = plt.subplots(1, len(test_epsilons)+1, figsize=(3*(len(test_epsilons)+1), 3))

# 原始样本
axes[0].imshow(test_img[0].cpu().permute(1,2,0).numpy())
axes[0].set_title(f'原始样本\n真实: {true_label}', color='green')
axes[0].axis('off')

# 不同epsilon的对抗样本
for i in range(len(test_epsilons)):
    img = adv_imgs_list[i][0].cpu().permute(1,2,0).numpy()
    pred_label = classes[preds_list[i]]
    color = 'green' if pred_label == true_label else 'red'
    
    axes[i+1].imshow(np.clip(img, 0, 1))
    axes[i+1].set_title(f'ε={test_epsilons[i]:.3f}\n预测: {pred_label}', color=color)
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig('../plots/demo_epsilon_effect.png', dpi=200)
plt.show()

---
## 4. 整体攻击成功率对比
快速计算三种攻击在干净模型和对抗训练模型上的成功率

In [ ]:
# 使用前1000个测试样本进行快速评估
subset_indices = list(range(1000))
subset_loader = DataLoader(Subset(testset, subset_indices), batch_size=32, shuffle=False)

print("=== 干净模型攻击成功率 ===")
asr_fgsm, _, _ = test_attack(model_clean, subset_loader, fgsm_attack, device, epsilon=epsilon)
print(f"FGSM ASR: {asr_fgsm:.2f}%")

asr_pgd, _, _ = test_attack(model_clean, subset_loader, pgd_attack, device, 
                            epsilon=epsilon, alpha=pgd_alpha, steps=pgd_steps)
print(f"PGD ASR: {asr_pgd:.2f}%")

asr_deepfool, _, _ = test_attack(model_clean, subset_loader, deepfool_attack, device,
                                max_iter=deepfool_max_iter, overshoot=deepfool_overshoot)
print(f"DeepFool ASR: {asr_deepfool:.2f}%")

print("\n=== 对抗训练模型攻击成功率 ===")
asr_fgsm_adv, _, _ = test_attack(model_adv, subset_loader, fgsm_attack, device, epsilon=epsilon)
print(f"FGSM ASR: {asr_fgsm_adv:.2f}%")

asr_pgd_adv, _, _ = test_attack(model_adv, subset_loader, pgd_attack, device, 
                                epsilon=epsilon, alpha=pgd_alpha, steps=pgd_steps)
print(f"PGD ASR: {asr_pgd_adv:.2f}%")

print("\n=== JPEG防御攻击成功率 ===")
from eval import test_jpeg_defense
asr_jpeg, def_acc_jpeg = test_jpeg_defense(model_clean, subset_loader, pgd_attack, device, 
                                           quality=jpeg_quality, epsilon=epsilon, alpha=pgd_alpha, steps=pgd_steps)
print(f"PGD + JPEG防御 ASR: {asr_jpeg:.2f}%")
print(f"JPEG防御后准确率: {def_acc_jpeg:.2f}%")

---
## 演示完成
所有生成的图片已保存至 `../plots/` 目录，完整实验结果请运行 `python eval.py` 查看。